# 🚀 ComfyUI + LTX 2.3 GGUF — Multi-Account Video Generator

**Dựa trên:** pogscafe/comfyui-kaggle-march-2025 (⭐2202) + Lightricks/ComfyUI-LTXVideo (⭐3.9k)

**Tính năng:** T2V, I2V, V2V, Motion Control trên Kaggle T4 GPU

**Model:** LTX-2.3 22B GGUF Q2_K (12.4 GB) — fits T4 16GB ✅

---
### 🔄 Cơ chế:
1. Notebook start → cài đặt ComfyUI + custom nodes
2. Download models
3. Start ComfyUI headless → Pinggy tunnel
4. Auto-register backend → Hermes dispatch job
5. Gen video → Hermes poll + download

In [ ]:
# ============================================================
# ⚙️ CONFIGURATION — Anh s?a cÃ¡c thÃ´ng s? nÃ y
# ============================================================

BACKEND_NAME = "kaggle-a"           # TÃªn backend (kaggle-a, kaggle-b, ...)
HERMES_PROXY_URL = "http://YOUR_SERVER_IP:9999"  # Ä?a ch? Hermes proxy
MODEL_CHOICE = "q2_k"               # q2_k (12GB), q3_k_s (14GB)
TUNNEL_CHOICE = "pinggy"            # pinggy hoáº·c cloudflare

print(f"✅ Config load: {BACKEND_NAME}")

---
## 1. CÃ i mÃ´i trÆ°á»?ng

In [ ]:
%%time
# === Fix venv Python 3.12 (theo cÃ¡ch pogscafe 2202 votes) ===
import os, sys, subprocess, shutil

home_dir = '/kaggle/working'
VENV_DIR = f'{home_dir}/venv'
COMFY_DIR = f'{home_dir}/ComfyUI'

os.chdir(home_dir)

# DÃ¹ng virtualenv thay vÃ¬ venv (fix Python 3.12)
!pip install -q virtualenv

if not os.path.exists(VENV_DIR):
    print('Táº¡o venv vá»?i Python 3.10...')
    subprocess.run(['virtualenv', VENV_DIR, '-p', '$(which python3.10)'], shell=True, check=False)
    # Fallback: dÃ¹ng python hiá»?n táº¡i
    if not os.path.exists(f'{VENV_DIR}/bin/python'):
        print('Fallback: dÃ¹ng pip --break-system-packages')
        USE_VENV = False
    else:
        USE_VENV = True
else:
    USE_VENV = os.path.exists(f'{VENV_DIR}/bin/python')

if USE_VENV:
    python = f'{VENV_DIR}/bin/python'
    pip = f'{VENV_DIR}/bin/pip'
else:
    python = sys.executable
    pip = 'pip3 install --break-system-packages'

print(f'Python: {python}')
print(f'USe venv: {USE_VENV}')

In [ ]:
%%time
# === Clone ComfyUI ===
%cd {home_dir}

if not os.path.exists(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    print('ComfyUI Ä?Ã£ cÃ³, pull update...')
    %cd {COMFY_DIR}
    !git pull

%cd {COMFY_DIR}
!{pip} install -q -r requirements.txt
print('âœ… ComfyUI ready')

In [ ]:
%%time
# === Clone custom nodes cho LTX 2.3 ===
%cd {COMFY_DIR}/custom_nodes

# Official Lightricks LTX-Video (3.9k sao)
if not os.path.exists('ComfyUI-LTXVideo'):
    !git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git
    print('âœ… ComfyUI-LTXVideo')

# LTXTricks - motion control
if not os.path.exists('ComfyUI-LTXTricks'):
    !git clone https://github.com/logtd/ComfyUI-LTXTricks.git
    print('âœ… ComfyUI-LTXTricks')

# ComfyUI-GGUF - load model GGUF
if not os.path.exists('ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git
    print('âœ… ComfyUI-GGUF')

print('âœ… All custom nodes installed')

---
## 2. Download Models

In [ ]:
%%time
# === Download LTX-2.3 GGUF Q2_K (12.4 GB) ===
# Source: QuantStack/LTX-2.3-GGUF — fits T4 16GB

model_dir = f'{COMFY_DIR}/models/unet'
os.makedirs(model_dir, exist_ok=True)
os.chdir(model_dir)

# Chá»?n quant: q2_k (12.4GB), q3_k_s (14GB), q3_k_m (14.7GB)
quant_map = {
    'q2_k': 'Q2_K',
    'q3_k_s': 'Q3_K_S',
    'q3_k_m': 'Q3_K_M',
}
quant = quant_map.get(MODEL_CHOICE, 'Q2_K')
model_file = f'LTX-2.3-distilled-1.1-{quant}.gguf'
model_url = f'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-{quant}.gguf'

if not os.path.exists(model_file):
    print(f'Download LTX-2.3 GGUF {quant} (~12 GB)...')
    !wget -c "{model_url}" -O "{model_file}" 2>&1
    print(f'âœ… Model done: {os.path.getsize(model_file)/1e9:.1f} GB')
else:
    print(f'âœ… Model already exists: {os.path.getsize(model_file)/1e9:.1f} GB')

In [ ]:
%%time
# === Download text encoder + VAE ===
# LTX-2 dÃ¹ng T5 text encoder (Gemma 2B)

clip_dir = f'{COMFY_DIR}/models/clip'
vae_dir = f'{COMFY_DIR}/models/vae'
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

# Text encoder (Gemma 2B)
if not os.path.exists(f'{clip_dir}/gemma-2b.safetensors'):
    !wget -c "https://huggingface.co/Lightricks/LTX-2/resolve/main/text_encoder/model.safetensors" \
        -O "{clip_dir}/gemma-2b.safetensors" 2>&1

# VAE
if not os.path.exists(f'{vae_dir}/ltx-vae.safetensors'):
    !wget -c "https://huggingface.co/Lightricks/LTX-2/resolve/main/vae/vae.safetensors" \
        -O "{vae_dir}/ltx-vae.safetensors" 2>&1

print('âœ… Text encoder + VAE ready')

---
## 3. Download Workflows

In [ ]:
%%time
# === Download workflows t? GitHub ===
import requests

workflow_dir = f'{COMFY_DIR}/user/default/workflows'
os.makedirs(workflow_dir, exist_ok=True)

# Workflows cÃ³ sáºµn trong repo (t? l?n tr??c)
github_base = 'https://raw.githubusercontent.com/damnquan07082004-hue/comfyui-kaggle-ltx23/main/sample-workflows'
workflows = [
    'ltx23_t2v.json',
    'ltx23_i2v.json',
    'ltx23_v2v.json',
    'ltx23_motion_control.json',
]

for wf in workflows:
    path = f'{workflow_dir}/{wf}'
    if not os.path.exists(path):
        try:
            r = requests.get(f'{github_base}/{wf}', timeout=10)
            with open(path, 'w') as f:
                f.write(r.text)
            print(f'âœ“ {wf}')
        except Exception as e:
            print(f'â?° {wf}: {e}')

print('âœ… Workflows ready')

---
## 4. Start ComfyUI + Tunnel

In [ ]:
# === Start ComfyUI headless ===
import subprocess

%cd {COMFY_DIR}

log_file = f'{home_dir}/comfyui.log'
with open(log_file, 'w') as f:
    proc = subprocess.Popen(
        [python, 'main.py', '--headless', '--port', '8188', '--listen', '127.0.0.1',
         '--dont-print-server', '--highvram'],
        stdout=f, stderr=f
    )

import time
time.sleep(10)

# Verify ComfyUI start
import requests
for i in range(30):
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'âœ… ComfyUI ready! PID: {proc.pid}')
            # Check if LTXVideo loaded
            if 'LTXVImgToVideo' in str(r.json().keys())[:200]:
                print('âœ… LTXVideo nodes loaded!')
            break
    except:
        pass
    time.sleep(3)
else:
    print('â?° ComfyUI may have issues, check log')

In [ ]:
# === Tunnel Pinggy ===
# theo cÃ¡ch pogscafe (2202 votes)

import subprocess, threading

def start_pinggy():
    cmd = f'ssh -p 443 -R0:localhost:8188 a.pinggy.io'
    proc = subprocess.Popen(
        cmd.split(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in proc.stdout:
        if 'https://' in line:
            url = line[line.find('https://'):].strip()
            if '.pinggy.link' in url or '.pinggy.io' in url:
                print(f'âœ… TUNNEL URL: {url}')
                # Save URL to file
                with open(f'{home_dir}/tunnel_url.txt', 'w') as f:
                    f.write(url)
                break

t = threading.Thread(target=start_pinggy, daemon=True)
t.start()

import time
time.sleep(15)
try:
    with open(f'{home_dir}/tunnel_url.txt') as f:
        TUNNEL_URL = f.read().strip()
        print(f'ðŸ”— Tunnel: {TUNNEL_URL}')
except:
    print('â?° Ch? tunnel... ch?y cell d??i ?? l?y url')

In [ ]:
# === N?u tunnel ch?a xu?t hi?n, ch?y cell nay ?? l?y ===
import time
for i in range(30):
    try:
        with open(f'{home_dir}/tunnel_url.txt') as f:
            TUNNEL_URL = f.read().strip()
            if TUNNEL_URL:
                print(f'ðŸ”— Tunnel URL: {TUNNEL_URL}')
                break
    except:
        pass
    time.sleep(5)
else:
    print('â?° Ch?a cÃ³ tunnel. Ki?m tra Pinggy log.')
    TUNNEL_URL = None

---
## 5. Register Backend v?i Hermes Proxy

In [ ]:
# === Auto-register backend ===
import requests, json

if TUNNEL_URL:
    payload = {
        'name': BACKEND_NAME,
        'url': TUNNEL_URL,
        'type': 'comfyui',
        'maxConns': 1,
        'capabilities': ['t2v', 'i2v', 'v2v', 'motion_control']
    }
    try:
        r = requests.post(f'{HERMES_PROXY_URL}/add-backend', json=payload, timeout=5)
        print(f'âœ… Registered backend: {r.status_code}')
    except Exception as e:
        print(f'â?° Cannot reach proxy: {e}')
        print(f'Backend info: {json.dumps(payload, indent=2)}')
else:
    print('â?° No tunnel URL yet')

---
## 6. Keep Alive

In [ ]:
# === Keep notebook alive, ch? job t? Hermes ===
from IPython.display import display, HTML, clear_output
import time, json

print(f'ðŸ”„ Backend: {BACKEND_NAME}')
print(f'ðŸ”— Tunnel: {TUNNEL_URL}')
print(f'âŒ©? Ch? job t? Hermes...')
print()
print('API:\n')
print(f'  POST {TUNNEL_URL}/prompt')
print(f'  GET  {TUNNEL_URL}/history/{{prompt_id}}')
print(f'  GET  {TUNNEL_URL}/view?filename={{output}}')

# Keep alive loop
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopped')